# Standalone ChunkFormer ingestion

All organizer videos and WAV files stay temporarily on Colab SSD. Transcript JSON and the SQLite index go to Drive. ChunkFormer is CC-BY-NC-4.0; confirm this is compatible with competition use.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install chunkformer
!mkdir -p /content/work


In [ ]:
import json
import re
import shutil
import sqlite3
import subprocess
from pathlib import Path

import torch
from chunkformer import ChunkFormerModel

work = Path("/content/work")
output = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
transcripts_dir = output / "asr"
transcripts_dir.mkdir(parents=True, exist_ok=True)
video_archives = [
    ("Videos_L21_a.zip", "https://aic-data.ledo.io.vn/Videos_L21_a.zip"),
    ("Videos_L22_a.zip", "https://aic-data.ledo.io.vn/Videos_L22_a.zip"),
    ("Videos_L23_a.zip", "https://aic-data.ledo.io.vn/Videos_L23_a.zip"),
    ("Videos_L24_a.zip", "https://aic-data.ledo.io.vn/Videos_L24_a.zip"),
    ("Videos_L25_a.zip", "https://aic-data.ledo.io.vn/Videos_L25_a.zip"),
    ("Videos_L26_a.zip", "https://aic-data.ledo.io.vn/Videos_L26_a.zip"),
    ("Videos_L26_b.zip", "https://aic-data.ledo.io.vn/Videos_L26_b.zip"),
    ("Videos_L26_c.zip", "https://aic-data.ledo.io.vn/Videos_L26_c.zip"),
    ("Videos_L26_d.zip", "https://aic-data.ledo.io.vn/Videos_L26_d.zip"),
    ("Videos_L26_e.zip", "https://aic-data.ledo.io.vn/Videos_L26_e.zip"),
    ("Videos_L27_a.zip", "https://aic-data.ledo.io.vn/Videos_L27_a.zip"),
    ("Videos_L28_a.zip", "https://aic-data.ledo.io.vn/Videos_L28_a.zip"),
    ("Videos_L29_a.zip", "https://aic-data.ledo.io.vn/Videos_L29_a.zip"),
    ("Videos_L30_a.zip", "https://aic-data.ledo.io.vn/Videos_L30_a.zip"),
]

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChunkFormerModel.from_pretrained("khanhld/chunkformer-ctc-large-vie").to(device)


In [ ]:
def timestamp_seconds(value):
    hours, minutes, seconds, milliseconds = map(int, value.split(":"))
    return hours * 3600 + minutes * 60 + seconds + milliseconds / 1000


def transcribe(video_path):
    audio_path = Path("/content") / f"{video_path.stem}.wav"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path), "-vn", "-ac", "1", "-ar", "16000", str(audio_path),
    ], check=True)
    result = model.endless_decode(
        audio_path=str(audio_path),
        chunk_size=64,
        left_context_size=128,
        right_context_size=128,
        total_batch_duration=14400,
        return_timestamps=True,
    )
    audio_path.unlink()
    segments = []
    for item in result:
        segments.append({
            "start": timestamp_seconds(item["start"]),
            "end": timestamp_seconds(item["end"]),
            "text": item["decode"],
        })
    return segments

for archive_name, url in video_archives:
    archive_path = work / archive_name
    extract_dir = work / archive_name.removesuffix(".zip")
    subprocess.run(["wget", "-q", url, "-O", str(archive_path)], check=True)
    subprocess.run(["unzip", "-q", str(archive_path), "-d", str(extract_dir)], check=True)
    video_paths = sorted(extract_dir.rglob("L*_V*.mp4"))
    for video_path in video_paths:
        transcript_path = transcripts_dir / f"{video_path.stem}.json"
        if transcript_path.exists():
            print(f"skip {video_path.stem}")
            continue
        segments = transcribe(video_path)
        transcript_path.write_text(json.dumps(segments, ensure_ascii=False), encoding="utf-8")
        print(f"saved {video_path.stem} {len(segments)} segments")
    shutil.rmtree(extract_dir)
    archive_path.unlink()


In [ ]:
database_path = output / "asr.sqlite"
if database_path.exists():
    database_path.unlink()

segments_count = 0
with sqlite3.connect(database_path) as connection:
    connection.execute("CREATE VIRTUAL TABLE asr USING fts5(video_id UNINDEXED, start_sec UNINDEXED, end_sec UNINDEXED, text)")
    for transcript_path in sorted(transcripts_dir.glob("L*_V*.json")):
        video_id = transcript_path.stem
        segments = json.loads(transcript_path.read_text(encoding="utf-8"))
        for segment in segments:
            connection.execute(
                "INSERT INTO asr(video_id, start_sec, end_sec, text) VALUES (?, ?, ?, ?)",
                (video_id, segment["start"], segment["end"], segment["text"]),
            )
            segments_count += 1
print(f"indexed {segments_count} segments")


In [ ]:
query = "người nói chuyện"
terms = []
for term in re.findall(r"\w+", query, flags=re.UNICODE):
    terms.append(f'"{term}"')
with sqlite3.connect(database_path) as connection:
    rows = connection.execute(
        "SELECT video_id, start_sec, end_sec, text, bm25(asr) FROM asr WHERE asr MATCH ? ORDER BY bm25(asr) LIMIT 5",
        (" OR ".join(terms),),
    ).fetchall()
for video_id, start, end, text, score in rows:
    print(video_id, start, end, text, score)
